# DistilBERT Intent Classifier

This notebook trains and evaluates a DistilBERT-based intent classifier
for the ResolveAI customer-support dataset.

The model is trained using the corrected leakage-safe generic-placeholder
training split and evaluated on the corresponding validation split.

The official test split remains untouched during model development.

After model selection, the frozen model will also be evaluated on the
independent OOD challenge benchmark to measure generalization to more
realistic customer language.

In [1]:
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [3]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Using device: cpu


### Model Configuration

In [4]:
MODEL_NAME = "distilbert-base-uncased"

TEXT_COLUMN = "processed_instruction"
TARGET_COLUMN = "intent"

### Load train and validation

In [5]:
train_df = pd.read_csv(
    "../data/processed/generic/train.csv"
)

validation_df = pd.read_csv(
    "../data/processed/generic/validation.csv"
)

print(
    "Train shape:",
    train_df.shape
)

print(
    "Validation shape:",
    validation_df.shape
)

Train shape: (18780, 9)
Validation shape: (4016, 9)


#### Verifying required columns

In [6]:
print(
    train_df.columns.tolist()
)

['flags', 'instruction', 'category', 'intent', 'response', 'normalized_instruction', 'split_group_key', 'processed_instruction', 'split']


In [7]:
required_columns = {
    TEXT_COLUMN,
    TARGET_COLUMN,
}

assert required_columns.issubset(
    train_df.columns
)

assert required_columns.issubset(
    validation_df.columns
)

print(
    "Required columns found."
)

Required columns found.


In [8]:
print(
    train_df[
        [
            TEXT_COLUMN,
            TARGET_COLUMN
        ]
    ]
    .isna()
    .sum()
)

print()

print(
    validation_df[
        [
            TEXT_COLUMN,
            TARGET_COLUMN
        ]
    ]
    .isna()
    .sum()
)

processed_instruction    0
intent                   0
dtype: int64

processed_instruction    0
intent                   0
dtype: int64
